# Notebook 01: MOSFET Physics Basics

**Objective:** Understand MOSFET device physics, the foundation of SPICE modeling.

We cover:
1. Physical constants and thermal voltage
2. Threshold voltage and body effect
3. Mobility models (field-dependent degradation)
4. MOSFET operating regions (cutoff, linear, saturation)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt

from src.device.physical import (
    PhysicalConstants, MobilityModel, ThresholdVoltage,
    thermal_voltage
)

## 1. Physical Constants

The `PhysicalConstants` dataclass holds all fundamental constants used in device physics.

In [ ]:
pc = PhysicalConstants()
print(f"Elementary charge q  = {pc.q:.3e} C")
print(f"Boltzmann constant k = {pc.k_B:.3e} J/K")
print(f"Thermal voltage V_T  = {pc.V_T*1000:.2f} mV (at {pc.T_nom:.1f} K)")
print(f"Si permittivity     = {pc.eps_si:.3e} F/m")
print(f"SiO2 permittivity   = {pc.eps_sio2:.3e} F/m")
print(f"Intrinsic ni         = {pc.n_i:.2e} m^-3")

## 2. Threshold Voltage

VTH depends on:
- **Fermi potential** φ_f: doping level
- **Body effect** γ: substrate bias modulates VTH
- **DIBL**: drain voltage lowers VTH in short channels

In [ ]:
tv = ThresholdVoltage()

# Fermi potential for N_sub = 1e16 cm^-3
n_sub = 1e22  # m^-3 = 1e16 cm^-3
phi_f = tv.compute_phi_f(n_sub)
print(f"Fermi potential phi_f = {phi_f*1000:.1f} mV")

# Gate oxide capacitance (TOX = 4 nm)
c_ox = tv.compute_c_ox(4e-9)
print(f"C_ox = {c_ox*1e3:.2f} mF/m^2")

# Body effect
gamma = tv.compute_gamma(n_sub, c_ox)
print(f"Gamma = {gamma:.3f} sqrt(V)")

# VTH vs V_SB
v_sb_range = np.linspace(0, 2, 100)
vth_vals = [tv.vth0_ideal(phi_f, gamma, v_sb) for v_sb in v_sb_range]

plt.figure(figsize=(6, 4))
plt.plot(v_sb_range, np.array(vth_vals))
plt.xlabel('V_SB (V)'); plt.ylabel('V_TH (V)')
plt.title('Body Effect: VTH vs Source-Body Bias')
plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Mobility Degradation

Carrier mobility decreases at high vertical fields due to surface roughness scattering.

In [ ]:
mm = MobilityModel(u0=400, theta=0.05)

vgs_range = np.linspace(0, 3, 200)
mu_vals = [mm.effective_mobility(vgs, 0.45, 0.1) for vgs in vgs_range]

plt.figure(figsize=(6, 4))
plt.plot(vgs_range, mu_vals)
plt.xlabel('V_GS (V)'); plt.ylabel('Mobility (cm^2/Vs)')
plt.title('Mobility Degradation with Gate Voltage')
plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 4. DIBL Effect

Drain-Induced Barrier Lowering reduces VTH at high V_DS.

In [ ]:
vds_range = np.linspace(0, 3, 200)
vth_dibl = [tv.vth_with_dibl(0.45, vds, eta0=0.05) for vds in vds_range]
vth_nodibl = [tv.vth_with_dibl(0.45, vds, eta0=0.0) for vds in vds_range]

plt.figure(figsize=(6, 4))
plt.plot(vds_range, vth_dibl, label='ETA0=0.05')
plt.plot(vds_range, vth_nodibl, '--', label='No DIBL')
plt.xlabel('V_DS (V)'); plt.ylabel('V_TH (V)')
plt.title('DIBL: VTH vs Drain Voltage')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Summary

- **Thermal voltage** kT/q ≈ 26 mV at 300K — sets subthreshold slope
- **Body effect** increases VTH when source is biased above body
- **DIBL** decreases VTH at high drain voltage (short-channel effect)
- **Mobility degradation** reduces current at high gate drive

Next: [Notebook 02 — I-V Curve Simulation](02_iv_curves.ipynb)